# 4.1 — Reader Policy and Corrupt Records

**Chapter 4, section 4.1.2 "Variety of Formats"**, and the starting point for **Exercise 1**.

**The question this notebook answers:** the chapter says a CSV reader must be given a policy for
lines it cannot parse, and that the three policies differ in *which consequence of being wrong*
you accept. What does each policy actually do, to each kind of broken record, on a real file?

The chapter also makes three claims about the audit query that counts the damage. All three are
surprising, all three are checkable, and all three are checked below.

**Data.** The course taxi file `taxi-data-sorted-verysmall.csv` — seventeen columns, no header —
with **five bad lines appended**, one per defect class. The broken copy is written to scratch;
nothing under `code/data/` is modified.

Runs on a laptop in about twenty seconds.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile, logging
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType, TimestampType)

DATA = os.environ.get("CS777_DATA", "../data")          # -> code/data/
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-4.1")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

# PySpark logs a structured record, JVM stack trace included, for every analysis error.
# This notebook catches several deliberately, so silence that logger.
for _lg in ("SQLQueryContextLogger", "DataFrameQueryContextLogger"):
    logging.getLogger(_lg).setLevel(logging.CRITICAL)

print("Spark", spark.version)
print("ANSI mode:", spark.conf.get("spark.sql.ansi.enabled"))

Spark 4.2.0
ANSI mode: true


## A file broken in five known ways

Every discussion of parse modes is vague until the defects are named, so we build the input
ourselves. A thousand good taxi records, then one line per defect class:

| Name | Defect | Field count |
|------|--------|-------------|
| `SHORT` | trailing fields missing | 12, not 17 |
| `LONG` | one surplus field | 18, not 17 |
| `BADNUM` | text where the schema declares a number | 17 |
| `SHIFT` | one field dropped and a numeric value appended, so every later value is one column to the left | 17 |
| `NASENT` | the string `NA` where a number belongs | 17 |

`SHIFT` is the one to watch. It is the defect the chapter says escapes every policy, and the
only way to see that is to build a shifted line whose values still *fit* the declared types.

In [2]:
lines = open(f"{DATA}/taxi-data-sorted-verysmall.csv").read().splitlines()
good = lines[:1000]
g = good[0].split(",")          # one real record, as a list of 17 fields

defects = {
    "SHORT":  ",".join(g[:12]),                          # 12 fields
    "LONG":   ",".join(g + ["EXTRA"]),                   # 18 fields
    "BADNUM": ",".join(g[:11] + ["NOTANUMBER"] + g[12:]),  # fare_amount is text
    # surcharge (field 13) dropped, a plausible number appended: still 17 fields,
    # every money column now holds its right-hand neighbour's value.
    "SHIFT":  ",".join(g[:12] + g[13:] + ["0.00"]),
    "NASENT": ",".join(g[:5] + ["NA"] + g[6:]),          # trip_distance is "NA"
}
by_line = {v: k for k, v in defects.items()}             # to name a flagged line later

BROKEN = os.path.join(SCRATCH, "taxi-broken.csv")
with open(BROKEN, "w") as f:
    f.write("\n".join(good + list(defects.values())) + "\n")

print(f"{len(good)} good records + {len(defects)} bad = {len(good) + len(defects)} lines")
print("written to", BROKEN)
print()
for name, line in defects.items():
    print(f"{name:7s} {len(line.split(',')):2d} fields  {line[:88]}")

1000 good records + 5 bad = 1005 lines
written to /var/folders/wh/ptq7zytj1gz42sqc0rs52tm40000gn/T/cs777/taxi-broken.csv

SHORT   12 fields  07290D3599E7A0D62097A346EFCC1FB5,E7750A37CAB07D0DFF0AF7E3573AC141,2013-01-01 00:00:00,20
LONG    18 fields  07290D3599E7A0D62097A346EFCC1FB5,E7750A37CAB07D0DFF0AF7E3573AC141,2013-01-01 00:00:00,20
BADNUM  17 fields  07290D3599E7A0D62097A346EFCC1FB5,E7750A37CAB07D0DFF0AF7E3573AC141,2013-01-01 00:00:00,20
SHIFT   17 fields  07290D3599E7A0D62097A346EFCC1FB5,E7750A37CAB07D0DFF0AF7E3573AC141,2013-01-01 00:00:00,20
NASENT  17 fields  07290D3599E7A0D62097A346EFCC1FB5,E7750A37CAB07D0DFF0AF7E3573AC141,2013-01-01 00:00:00,20


## The declared schema, and the column that keeps the evidence

The chapter's listing declares all seventeen fields and appends an eighteenth, a string column
named by `columnNameOfCorruptRecord`. That column costs nothing when a record parses — it is
null — and holds the original raw line when it does not.

Two schemas are built here: one *with* the corrupt-record column, for `PERMISSIVE` reads, and
one without, because `DROPMALFORMED` and `FAILFAST` have no evidence to keep.

In [3]:
fields = [
    StructField("medallion",         StringType(),    True),
    StructField("hack_license",      StringType(),    True),
    StructField("pickup_datetime",   TimestampType(), True),
    StructField("dropoff_datetime",  TimestampType(), True),
    StructField("trip_time",         IntegerType(),   True),   # seconds
    StructField("trip_distance",     DoubleType(),    True),   # miles
    StructField("pickup_longitude",  DoubleType(),    True),
    StructField("pickup_latitude",   DoubleType(),    True),
    StructField("dropoff_longitude", DoubleType(),    True),
    StructField("dropoff_latitude",  DoubleType(),    True),
    StructField("payment_type",      StringType(),    True),
    StructField("fare_amount",       DoubleType(),    True),
    StructField("surcharge",         DoubleType(),    True),
    StructField("mta_tax",           DoubleType(),    True),
    StructField("tip_amount",        DoubleType(),    True),
    StructField("tolls_amount",      DoubleType(),    True),
    StructField("total_amount",      DoubleType(),    True),
]
CORRUPT = "_corrupt_record"
schema_audited = StructType(fields + [StructField(CORRUPT, StringType(), True)])
schema_plain   = StructType(fields)


def read(mode, audited=True, **options):
    """The chapter's reader call, with the mode and any extra options supplied."""
    opts = dict(header="false", sep=",", mode=mode)
    if audited:
        opts["columnNameOfCorruptRecord"] = CORRUPT
    opts.update(options)
    return (spark.read.format("csv")
            .schema(schema_audited if audited else schema_plain)
            .options(**opts)
            .load(BROKEN))

print(f"{len(fields)} declared fields + the corrupt-record column")

17 declared fields + the corrupt-record column


## What `PERMISSIVE` does with each defect

The default policy keeps every row. Four of our five lines are flagged; one is not.

In [4]:
parsed = read("PERMISSIVE").cache()      # the cache matters -- see the next section
print("rows returned:", parsed.count())

flagged = [by_line.get(r[0], "(a good record?!)")
           for r in parsed.where(F.col(CORRUPT).isNotNull()).select(CORRUPT).collect()]
print("flagged as corrupt:", sorted(flagged))
print("not flagged       :", sorted(set(defects) - set(flagged)))

rows returned: 1005
flagged as corrupt: ['BADNUM', 'LONG', 'NASENT', 'SHORT']
not flagged       : ['SHIFT']


In [5]:
# The parsed values of the five defect lines, in file order, beside a good record.
tail = (parsed.select("trip_time", "trip_distance", "payment_type",
                      "fare_amount", "surcharge", "total_amount",
                      F.col(CORRUPT).isNotNull().alias("corrupt"))
        .tail(6))
print(f"{'':8s} {'trip_time':>9s} {'distance':>9s} {'pay':>4s} {'fare':>6s} "
      f"{'surch':>6s} {'total':>7s}  corrupt")
for name, r in zip(["(good)"] + list(defects), tail):
    print(f"{name:8s} {str(r[0]):>9s} {str(r[1]):>9s} {str(r[2]):>4s} {str(r[3]):>6s} "
          f"{str(r[4]):>6s} {str(r[5]):>7s}  {r[6]}")

         trip_time  distance  pay   fare  surch   total  corrupt
(good)         240      0.75  CSH    5.0    0.5     6.0  False
SHORT          120      0.44  CSH    3.5   None    None  True
LONG           120      0.44  CSH    3.5    0.5     4.5  True
BADNUM         120      0.44  CSH   None    0.5     4.5  True
SHIFT          120      0.44  CSH    3.5    0.5     0.0  False
NASENT         120      None  CSH    3.5    0.5     4.5  True


Read that table one row at a time, because each line is a different lesson.

* **`SHORT`** came back at full width with its missing trailing columns null. The shape was
  repaired quietly, and the row was *also* flagged.
* **`LONG`** came back with the surplus field discarded — and flagged.
* **`BADNUM`** kept every other value and nulled only `fare_amount` — and flagged.
* **`NASENT`** is flagged too, because `NA` is not a number and nothing has told the reader
  otherwise. The next-to-last section fixes that at the reader, which is where it is cheapest.
* **`SHIFT` is not flagged.** Its seventeen values all fit their declared types, so nothing is
  malformed. Look at what it means, though: `surcharge` holds the value that belonged to
  `mta_tax`, and `total_amount` is the `0.00` we appended. The record parses perfectly and is
  wrong in every money column after the twelfth.

That is the chapter's central qualification, visible in one row: **the parse mode governs
whether a record can be read, never whether it means what its position claims.**

## Taking the audit number is harder than writing it

The chapter makes three claims about the query that counts corrupt records. Each one is a way
of getting a wrong number without any error being raised, so each is worth reproducing.

In [6]:
BAD  = F.count(F.when(F.col(CORRUPT).isNotNull(), 1)).alias("bad")
ROWS = F.count("*").alias("rows")

def attempt(label, fn):
    spark.catalog.clearCache()            # each attempt must start from the raw file
    try:
        print(f"{label:52s} -> {fn()}")
    except Exception as e:
        condition = getattr(e, "getCondition", lambda: type(e).__name__)()
        print(f"{label:52s} -> REFUSED: {condition}")

# (a) the corrupt-record column, on its own
attempt("the corrupt column alone",
        lambda: len(read("PERMISSIVE").select(CORRUPT).limit(3).collect()))
attempt("bad, with count(*) only",  lambda: read("PERMISSIVE").select(BAD, ROWS).first())

# (b) the same audit, pruned to a handful of data columns
attempt("bad + count(medallion)",
        lambda: read("PERMISSIVE").select(BAD, F.count("medallion").alias("n")).first())
attempt("bad + count(fare_amount)",
        lambda: read("PERMISSIVE").select(BAD, F.count("fare_amount").alias("n")).first())

# (c) materialize first, then audit
def audited():
    spark.catalog.clearCache()
    p = read("PERMISSIVE").cache()
    p.count()                             # the action that fills the cache
    return p.select(BAD, F.count("medallion").alias("n")).first()
print(f"{'CACHED first, then bad + count(medallion)':52s} -> {audited()}")

the corrupt column alone                             -> REFUSED: UNSUPPORTED_FEATURE.QUERY_ONLY_CORRUPT_RECORD_COLUMN


bad, with count(*) only                              -> REFUSED: UNSUPPORTED_FEATURE.QUERY_ONLY_CORRUPT_RECORD_COLUMN
bad + count(medallion)                               -> Row(bad=0, n=1005)


bad + count(fare_amount)                             -> Row(bad=1, n=1004)


CACHED first, then bad + count(medallion)            -> Row(bad=4, n=1005)


The true number is **four**. The three failures above produce it as follows.

1. **A query whose surviving raw-file columns are only the corrupt-record column is refused
   outright**, with `UNSUPPORTED_FEATURE.QUERY_ONLY_CORRUPT_RECORD_COLUMN`. Note that the
   second attempt *looks* like it reads two things, but `count(*)` reads no column at all, so
   after column pruning only the corrupt column is left and the refusal applies. The error text
   names its own remedy: cache or save the parsed result, then query that.
2. **A pruned query is worse than refused: it runs and under-reports.** Asking for the corrupt
   count beside `count(medallion)` returns **zero** corrupt records, and beside
   `count(fare_amount)` it returns **one** — the one line whose `fare_amount` could not be
   parsed. Neither query is wrong about what it was asked; the lines were simply never parsed
   widely enough for the other failures to be seen.
3. **Materializing first gives the right answer.** `cache()` here, or the write that a bulk
   load performs anyway.

The second failure is the dangerous one, because a zero in a monitoring dashboard reads as
*clean input*.

## `count()` is not a parse

One more result looks like a contradiction until the reason is clear: the row count of a file
full of corruption is the same under every policy, `FAILFAST` included.

In [7]:
for mode in ["PERMISSIVE", "DROPMALFORMED", "FAILFAST"]:
    spark.catalog.clearCache()
    print(f"{mode:14s} count()          -> {read(mode, audited=False).count():,}")

PERMISSIVE     count()          -> 1,005
DROPMALFORMED  count()          -> 1,005


FAILFAST       count()          -> 1,005


In [8]:
# ... but an action that needs the parsed values behaves as the policy promises.
spark.sparkContext.setLogLevel("FATAL")      # the FAILFAST failure is raised on an executor
spark.catalog.clearCache()
print("DROPMALFORMED, collect():", len(read("DROPMALFORMED", audited=False).collect()), "rows")
try:
    read("FAILFAST", audited=False).collect()
    print("FAILFAST, collect(): no error (unexpected)")
except Exception as e:
    import re
    text = str(e)
    print("FAILFAST, collect(): raised", type(e).__name__)
    for condition in list(re.finditer(r"\[[A-Z_]+(?:\.[A-Z_]+)?\]", text))[:2]:
        print("   ", " ".join(text[condition.start():condition.start() + 130].split()))
finally:
    spark.sparkContext.setLogLevel("ERROR")

DROPMALFORMED, collect(): 1001 rows


FAILFAST, collect(): raised Py4JJavaError
    [FAILED_READ_FILE.NO_HINT] Encountered error while reading file file:///var/folders/wh/ptq7zytj1gz42sqc0rs52tm40000gn/T/cs777/taxi
    [MALFORMED_RECORD_IN_PARSING.WITHOUT_SUGGESTION] Malformed records are detected in record parsing: [07290D3599E7A0D62097A346EFCC1F


`count()` returns 1,005 — the number of **lines** — under all three policies, because Spark
answers it without materializing a single parsed column, so no record is ever offered to the
parser for it to object to. `collect()` is a different matter: `DROPMALFORMED` returns 1,001
rows, having discarded all four malformed lines, and `FAILFAST` stops the job.

The practical consequence: **a row count is not a validation.** It tells you how many lines
arrived, not how many records were usable.

## The cheapest repair happens at the reader

`NASENT` was flagged as corrupt only because nothing told the reader what `NA` means. The
`nullValue` option is one string in the reader call, it applies to every column type, and it
turns a corrupt record into an honest null.

In [9]:
for label, options in [("without nullValue", {}), ("with nullValue='NA'", {"nullValue": "NA"})]:
    spark.catalog.clearCache()
    p = read("PERMISSIVE", **options).cache()
    p.count()
    names = sorted(by_line.get(r[0], "?")
                   for r in p.where(F.col(CORRUPT).isNotNull()).select(CORRUPT).collect())
    honest = p.where(F.col("trip_distance").isNull() & F.col(CORRUPT).isNull()).count()
    print(f"{label:22s} corrupt: {names}")
    print(f"{'':22s} rows with a null trip_distance and NO corruption flag: {honest}")

without nullValue      corrupt: ['BADNUM', 'LONG', 'NASENT', 'SHORT']
                       rows with a null trip_distance and NO corruption flag: 0


with nullValue='NA'    corrupt: ['BADNUM', 'LONG', 'SHORT']
                       rows with a null trip_distance and NO corruption flag: 1


With `nullValue="NA"` the sentinel parses to `null`, the record is no longer corrupt, and the
missingness has become something the next section of the chapter can treat — a null in
`trip_distance` rather than a whole unparseable line. Sentinels that are *valid numbers*, such
as `-1`, cannot be caught this way; they have to be found by profiling, which is notebook
[4.7](04.07%20Data%20Validation%20and%20Profiling.ipynb).

## The policy table, and Exercise 1

Everything above, as one table. "Not flagged" in the last column is the reason the chapter
insists that a parse mode is not a validation.

| | `SHORT` / `LONG` (wrong field count) | `BADNUM` / `NASENT` (bad value) | `SHIFT` (column shift) |
|---|---|---|---|
| `PERMISSIVE` | row kept at full width, shape repaired, **flagged** | row kept, bad field null, **flagged** | row kept, **not flagged** |
| `DROPMALFORMED` | row discarded, nothing reported | row discarded, nothing reported | row kept, **not flagged** |
| `FAILFAST` | job stops | job stops | row kept, **not flagged** |

Exercise 1 asks for a mode per scenario. The answers this notebook supports:

* **(a) nightly bulk load, small expected error rate that must be quantified** — `PERMISSIVE`
  with a declared corrupt-record column, plus the *cached* audit. Nothing is lost, the raw text
  of every failure is queryable, and the count decides whether the batch is usable.
* **(b) one-time migration of certified-clean data** — `FAILFAST`. Corruption means an upstream
  bug, and a loud stop is the cheapest possible outcome.
* **(c) exploratory read of an undocumented file** — `PERMISSIVE` with a corrupt-record column,
  because on first contact the malformed lines are the most informative rows in the file.
* **(d)** With no corrupt-record column declared, `PERMISSIVE` still returns the row at full
  width with unparseable fields null. What is lost is the *evidence*: nothing distinguishes a
  field that failed to parse from one that was legitimately empty.
* **(e)** A colleague whose `FAILFAST` read raised nothing, yet whose rows show a fare in the
  surcharge column, has the `SHIFT` defect. No mode catches it, because nothing about it is
  malformed. A **range check** catches it — the shifted row above carries `total_amount` of
  `0.00` against a `fare_amount` of `3.50`, which no honest trip does — and so does a schema
  check only if the shift moves a value across a type boundary, which is exactly what one
  cannot count on.

In [10]:
spark.catalog.clearCache()
print("scratch:", BROKEN)

scratch: /var/folders/wh/ptq7zytj1gz42sqc0rs52tm40000gn/T/cs777/taxi-broken.csv


## Conclusion

The chapter's argument, now measured on a file we broke on purpose:

* **A wrong field count is corruption like any other**, and all three policies act on it:
  `PERMISSIVE` flags it, `DROPMALFORMED` discards the line, `FAILFAST` stops.
* **A column shift is not corruption at all** as far as the reader is concerned, and it is the
  one defect that survives every policy and changes every answer.
* **`PERMISSIVE` with a declared corrupt-record column is the only policy that keeps the
  evidence**, and the evidence is worth nothing until it is counted — from a materialized
  result, because the audit query is otherwise either refused or silently pruned to zero.
* **`nullValue` is cheaper than everything downstream.** One reader option turned a corrupt
  record into a null, where the cleaning techniques of §4.2 can deal with it.

Next: [4.2](04.02%20Cleaning%20the%20Taxi%20Data.ipynb) takes the records that survive
ingestion and repairs them.